In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 11 — Reference coding solutions

These are instructor reference implementations, not evidence of your independent mastery. Run after attempting your own version. All examples use explicit small inputs and the declared dataset partitions.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
from itertools import combinations
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/manifest.json').exists() and (p/'src/shape_lab').exists()),None)
if ROOT is None: raise RuntimeError('Open this notebook inside the extracted course folder.')
sys.path.insert(0,str(ROOT/'src'))
from shape_lab.data import load_iris_data,load_digit_data,digit_example
from shape_lab.algebra import closure,validate_complex,rank_mod2,boundary_matrix,betti_numbers
from shape_lab.topology import component_labels
from shape_lab.persistence import ordered_filtration,rips_filtration,reduce_boundary,persistent_homology,diagram,pixel_filtration,finite_bottleneck
print('Working inside',ROOT.name,'— no network data downloads.')

Working inside course — no network data downloads.


## 11.C1 — Build a Betti curve

For finite or infinite birth/death pairs and a grid, return the number of active intervals at every grid value.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Apply the same alive-count rule repeatedly.

</details>

In [3]:
def betti_curve(pairs,grid):
    return np.array([sum(b<=t<d for b,d in pairs) for t in grid],dtype=int)

In [4]:
assert np.array_equal(betti_curve([(0,1),(.5,2)],[0,.5,1,2]),[1,2,1,0])
image,sample_id=digit_example(0)
D=diagram(persistent_homology(pixel_filtration(1-image/16)),1,finite_only=True)
print("Real image H1 Betti curve:",betti_curve(D,np.linspace(0,1,5)))

Real image H1 Betti curve: [0 0 1 1 0]


**Why this works and what it does not establish:** The curve is a summary: it forgets some pairings retained by the complete diagram. Equal Betti curves do not force identical barcodes.

## 11.C2 — Build the first persistence landscape

For a finite diagram and grid return the largest tent value at each grid point, with zeros for an empty diagram.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

A tent rises from birth, falls toward death, and is zero outside the interval.

</details>

In [5]:
def first_landscape(diagram_,grid):
    grid=np.asarray(grid,dtype=float);out=np.zeros(len(grid))
    for birth,death in diagram_:
        tent=np.maximum(0,np.minimum(grid-birth,death-grid))
        out=np.maximum(out,tent)
    return out

In [6]:
grid=np.array([0,.5,1,1.5,2])
assert np.allclose(first_landscape([(0,2)],grid),[0,.5,1,.5,0])
assert np.allclose(first_landscape([],grid),0)
print("Landscape values:",first_landscape([(0,2)],grid))

Landscape values: [0.  0.5 1.  0.5 0. ]


**Why this works and what it does not establish:** Higher landscape levels use the second-largest, third-largest, and later tent values. Taking only the first level discards information about overlapping intervals.

## 11.C3 — Fit scaling only on training data

Given train and other matrices with nonconstant training columns, return both standardized using the training column mean and population standard deviation.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

Do not calculate a second mean from the other partition.

</details>

In [7]:
def train_scaled(train,other):
    train=np.asarray(train,dtype=float);other=np.asarray(other,dtype=float)
    mean=train.mean(axis=0);std=train.std(axis=0)
    return (train-mean)/std,(other-mean)/std

In [8]:
A,B=train_scaled([[0],[2]],[[100]])
assert np.allclose(A,[[-1],[1]]) and np.allclose(B,[[99]])
X=load_iris_data().iloc[:,1:5].to_numpy();A,B=train_scaled(X[:90],X[90:])
assert np.allclose(A.mean(axis=0),0,atol=1e-12)
print("Other-partition means need not be zero:",B.mean(axis=0))

Other-partition means need not be zero: [ 1.48222733 -0.43097143  1.76404064  2.02688437]


**Why this works and what it does not establish:** The hand-picked Iris split here is only a preprocessing demonstration, not a representative classifier evaluation. A validation or test transformation can have nonzero means without indicating a bug.

## 11.C4 — Attach availability times to delay windows

For a one-dimensional series, positive dimension m and lag, return forward windows and the last input index used by each. Assume enough observations.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

The first index in the window is not the time at which all its information was known.

</details>

In [9]:
def windows_with_availability(series,m,lag):
    x=np.asarray(series);n=len(x)-(m-1)*lag
    windows=np.column_stack([x[j*lag:j*lag+n] for j in range(m)])
    return windows,np.arange(n)+(m-1)*lag

In [10]:
W,t=windows_with_availability(np.arange(7),3,2)
assert np.array_equal(W,[[0,2,4],[1,3,5],[2,4,6]])
assert np.array_equal(t,[4,5,6])
print("Forward windows become available at:",t)

Forward windows become available at: [4 5 6]


**Why this works and what it does not establish:** Using a window as though it were available at its first sample introduces future information. The coordinate construction alone does not establish the hypotheses of a dynamical reconstruction theorem.

## 11.C5 — Choose a model on validation only

Given rows containing C and validation_accuracy, choose the largest validation accuracy, breaking ties toward the smaller C.

**Before running:** describe the input type, output type, and a hand-checkable expected result.

<details><summary>Hint</summary>

A minus sign sorts accuracy from high to low while C sorts from low to high.

</details>

In [11]:
def select_validation(rows):
    return sorted(rows,key=lambda r:(-r['validation_accuracy'],r['C']))[0]['C']

In [12]:
rows=[{'C':10.,'validation_accuracy':.9},{'C':1.,'validation_accuracy':.9},{'C':.1,'validation_accuracy':.8}]
assert select_validation(rows)==1.
print("Declared tie-break selected C=1 without a test score")

Declared tie-break selected C=1 without a test score


**Why this works and what it does not establish:** Tie-breaking rules should be declared before using test results. Once test outcomes have been inspected, tuning based on them no longer produces a fresh independent holdout evaluation.